In [40]:
import pandas as pd
from build_dataset import build_dataset

In [41]:
# ── Load all stations ─────────────────────────────────────────────────────────
all_stations = pd.read_csv(
    "all_stations.csv",
    header=None,
    names=["name", "station_id", "data_owner", "latitude", "longitude"],
)
all_stations["latitude"]  = pd.to_numeric(all_stations["latitude"],  errors="coerce")
all_stations["longitude"] = pd.to_numeric(all_stations["longitude"], errors="coerce")
all_stations = all_stations.dropna(subset=["latitude", "longitude"])

print(f"Total stations: {len(all_stations):,}")

Total stations: 8,169


In [42]:
# ── Bounding box (adjust if needed) ──────────────────────────────────────────
BBOX = {
    "lat_min": -34.80,
    "lat_max": -33.20,
    "lon_min":  149.90,
    "lon_max":  151.60,
}

sydney_stations = all_stations[
    all_stations["latitude"].between(BBOX["lat_min"], BBOX["lat_max"]) &
    all_stations["longitude"].between(BBOX["lon_min"], BBOX["lon_max"])
].reset_index(drop=True)

print(f"Stations inside bbox: {len(sydney_stations)}")
sydney_stations

Stations inside bbox: 425


,name,station_id,data_owner,latitude,longitude
0,10A,GW040994,NSW - Water NSW,-34.483856,150.566715
1,11A,GW040997,NSW - Water NSW,-34.526630,150.564622
2,1A,GW040955,NSW - Water NSW,-34.512654,150.524219
3,1M,GW040996,NSW - Water NSW,-34.511769,150.523349
4,1M3d,GW075176,NSW - Water NSW,-34.517254,150.523986
...,...,...,...,...,...
420,YARRAWARRAH,566056,NSW - Sydney Water Corporation (Sydney Water),-34.057659,151.034028
421,YERRANDERIE,563055,NSW - Water NSW,-34.116667,150.219166
422,YERRANDERIE (BINDOOK,563057,NSW - Water NSW,-34.171789,150.095167
423,YERRANDERIE-BYRNESCK,563036,NSW - Water NSW,-34.138700,150.311994


In [18]:
# ── Keep only numeric station IDs (BOM rainfall network) ─────────────────────
# GW*, PI_* etc. are groundwater/water level stations — the BOM SOS API
# only returns rainfall data for plain numeric IDs.
def is_numeric_id(sid):
    try:
        int(str(sid))
        return True
    except ValueError:
        return False

rainfall_stations = sydney_stations[
    sydney_stations["station_id"].apply(is_numeric_id)
].reset_index(drop=True)

print(f"Stations inside bbox : {len(sydney_stations)}")
print(f"Rainfall stations    : {len(rainfall_stations)}  (numeric IDs only)")
print(f"Dropped              : {len(sydney_stations) - len(rainfall_stations)}  (GW*, PI_*, etc.)")
print()
print(rainfall_stations[["station_id", "name", "data_owner"]].to_string(index=False))

STATION_IDS = rainfall_stations["station_id"].astype(str).tolist()

Stations inside bbox : 425
Rainfall stations    : 354  (numeric IDs only)
Dropped              : 71  (GW*, PI_*, etc.)

station_id                 name                                    data_owner
    568171       ALBION PARK BC NSW - Sydney Water Corporation (Sydney Water)
   2122341              ALCORNS                               NSW - Water NSW
    567105           ANNANGROVE NSW - Sydney Water Corporation (Sydney Water)
    568167 APPIN (INGHAMS) TBRG                               NSW - Water NSW
    566172            AVALON BC NSW - Sydney Water Corporation (Sydney Water)
    212211             AVON DAM                               NSW - Water NSW
    568046             AVON DAM                               NSW - Water NSW
    212210  AVON R. @ AVON WEIR                               NSW - Water NSW
    568057      AVON TOWER TBRG                               NSW - Water NSW
    568162  BALGOWNIE RESERVOIR NSW - Sydney Water Corporation (Sydney Water)
    212065     BARABA@

In [ ]:
df = build_dataset(
    station_ids=STATION_IDS,
    start_date="2021-01-01",
    end_date="2025-12-31",
    output_dir="database/australia/raw",
    combined_output="all_stations_rainfall_hourly_combined.csv",
)

Fetching 354 stations  |  2021-01-01 → 2025-12-31
[  1/354] 568171  ✓  166973 records → 4609 rainy hours  |  92545.5 mm
[  2/354] 2122341    [!] No data returned
[  3/354] 567105  ✓  128004 records → 3635 rainy hours  |  64007.0 mm
[  4/354] 568167  ✓  109971 records → 3592 rainy hours  |  60081.6 mm
[  5/354] 566172  ✓  155941 records → 4674 rainy hours  |  85088.5 mm
[  6/354] 212211    [!] No data returned
[  7/354] 568046  ✓  124167 records → 4110 rainy hours  |  79297.0 mm
[  8/354] 212210    [!] No data returned
[  9/354] 568057  ✓   64678 records → 2834 rainy hours  |  61988.0 mm
[ 10/354] 568162  ✓  177006 records → 4515 rainy hours  |  100463.0 mm
[ 11/354] 212065    [!] No data returned
[ 12/354] 568042    [!] No data returned
[ 13/354] 568128  

In [34]:
combined = pd.read_csv("./all_stations_rainfall_hourly_combined.csv")

In [35]:
combined["rainfall_mm"].describe()

count    8.238912e+06
mean     1.735136e+01
std      4.319493e+03
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.151900e+07
Name: rainfall_mm, dtype: float64

In [36]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd

r = requests.get("https://www.bom.gov.au/waterdata/services", params={
    "service": "SOS",
    "version": "2.0",
    "request": "GetObservation",
    "observedProperty": "http://bom.gov.au/waterdata/services/parameters/Rainfall",
    "featureOfInterest": "http://bom.gov.au/waterdata/services/stations/568171",
    "temporalFilter": "om:phenomenonTime,2021-01-01/2021-01-07",
})

NS = {"wml2": "http://www.opengis.net/waterml/2.0"}
root = ET.fromstring(r.content)

times, values = [], []
for point in root.findall(".//wml2:MeasurementTVP", NS):
    t = point.find("wml2:time", NS)
    v = point.find("wml2:value", NS)
    if t is not None and v is not None:
        times.append(t.text)
        values.append(v.text)

df = pd.DataFrame({"timestamp": times, "rainfall_mm": values})
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df["rainfall_mm"] = pd.to_numeric(df["rainfall_mm"])


df["rainfall_mm"].value_counts()


rainfall_mm
0.5       1758
0.0        150
14.0         2
11.0         1
32.5         1
21.0         1
20.0         1
44.0         1
13.0         1
7.0          1
164.0        1
1208.5       1
Name: count, dtype: int64